# Evaluación Parcial 1 — Machine Learning
## CRISP-DM: Comprensión del negocio, comprensión de los datos y preparación de los datos

**Dataset:** partidas y rondas de Counter-Strike.


### Objetivo
Desarrollar las tres primeras fases de CRISP-DM mediante un análisis reproducible y justificado. Las decisiones de limpieza, imputación, tratamiento de outliers, encoding y escalamiento se tomarán a partir de la evidencia observada en los datos.

## 0. Preguntas de investigación preliminares

### Pregunta principal de clasificación
**¿Cómo cambia la probabilidad de ganar una partida según el desempeño del equipo en las primeras rondas de cada lado (Terrorist y Counter-Terrorist), considerando además el mapa y la economía disponible?**

- **Target de clasificación:** `MatchWinner`.
- **Unidad analítica futura:** equipo-partida.
- **Momento de predicción:** después de observar las primeras rondas de ambos lados, utilizando solamente información disponible hasta ese momento.

### Pregunta principal de regresión
**¿Es posible estimar cuánto tiempo permanecerá con vida un jugador durante una ronda a partir de información disponible al comienzo de la misma?**

- **Target de regresión:** `TimeAlive`.
- **Unidad analítica futura:** jugador-ronda.
- `TimeAlive` es ahora una variable numérica continua interpretable, por lo que constituye un target de regresión adecuado.
- Variables derivadas del propio tiempo de supervivencia o generadas durante/después de la ronda deberán excluirse para evitar *data leakage*.

### Regla metodológica
La clasificación y la regresión tienen granularidades distintas. Por ello, se prepararán dos conjuntos analíticos separados en la Fase 3, evitando mezclar registros jugador-ronda con observaciones equipo-partida.

# 1. Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import trim_mean

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Durante el desarrollo se puede usar "head.csv".
# Para la entrega, reemplazar por la ruta del dataset completo con el mismo esquema.
DATA_PATH = "head.csv"

df_raw = pd.read_csv("demo_round_traces.csv", sep=";")

print("Dimensiones:", df_raw.shape)
display(df_raw.head())

# FASE 1 — COMPRENSIÓN DEL NEGOCIO

## 1.1 Identificación del problema

Counter-Strike se organiza en partidas compuestas por rondas. Durante la partida, los equipos alternan entre los lados Terrorist y CounterTerrorist. El desempeño temprano puede generar ventajas competitivas y económicas que potencialmente afectan el resultado final.

El proyecto busca estudiar cuánto aporta el desempeño de las primeras rondas a la explicación y eventual predicción del resultado final de una partida.

## 1.2 Objetivos

### Objetivo general
Analizar los factores asociados al resultado de una partida y preparar un conjunto de datos apto para tareas supervisadas de clasificación y regresión.

### Objetivos específicos
1. Describir estadísticamente las variables del dataset.
2. Identificar problemas de calidad, valores faltantes, duplicados y valores atípicos.
3. Analizar la relación entre mapa, lado, rondas iniciales, economía y resultado.
4. Definir un target para clasificación y otro para regresión.
5. Determinar qué técnicas de imputación, encoding y escalamiento son apropiadas.
6. Construir un dataset preparado para una futura fase de modelado.

## 1.3 Criterios de éxito preliminares

- Obtener variables correctamente tipadas y documentadas.
- Justificar el tratamiento de missing values y outliers.
- Evitar utilizar identificadores o información futura como variables predictoras.
- Construir variables derivadas coherentes con la pregunta de investigación.

### Conclusión Fase 1 — Comprensión del negocio

El análisis propuesto busca determinar cómo el desempeño inicial y las condiciones económicas de los equipos influyen en el desarrollo de una partida de Counter-Strike y en la probabilidad de obtener la victoria. En particular, se analizará la importancia de las primeras rondas jugadas tanto del lado Terrorist como Counter-Terrorist, considerando que el resultado de estas rondas puede afectar la economía disponible, las decisiones estratégicas posteriores y, eventualmente, el resultado final de la partida.

El conocimiento obtenido puede ser útil para los jugadores, ya que permitirá cuantificar mediante datos la importancia de comenzar favorablemente cada lado de una partida. Esto puede ayudar a comprender mejor el impacto de las decisiones tomadas durante las primeras rondas y apoyar una preparación estratégica más informada.

Como objetivo de **clasificación** se establece `MatchWinner`, que indica si el equipo obtuvo la victoria final. El análisis predictivo se centrará en dos momentos relevantes: el inicio del lado Terrorist y el inicio del lado Counter-Terrorist. Se construirán variables que representen el rendimiento del equipo en las primeras rondas de ambos lados y, cuando corresponda, su situación económica.

Como objetivo de **regresión** se establece `TimeAlive`. En el dataset esta variable se encuentra representada como un valor numérico continuo y posee una interpretación directa en términos de tiempo de supervivencia durante la ronda. La regresión se planteará a nivel jugador-ronda y buscará estimar `TimeAlive` utilizando únicamente información disponible al comienzo de la ronda.

Las dos tareas requieren controlar explícitamente el *data leakage*. Para predecir `MatchWinner` no se utilizarán estadísticas finales de la partida ni información de rondas posteriores al instante de predicción. Para predecir `TimeAlive` tampoco se utilizarán variables derivadas del propio tiempo de supervivencia ni eventos que ocurren durante o después de la ronda.

De esta manera, el proyecto mantiene una misma temática de análisis del desempeño en Counter-Strike, pero diferencia correctamente dos problemas supervisados: clasificación a nivel equipo-partida y regresión a nivel jugador-ronda.

# FASE 2 — COMPRENSIÓN DE LOS DATOS

Se observan **37 columnas**. Entre los campos aparecen:

- `SteamId`: identificador del jugador.
- `ScaledTimeAlive`: versión escalada de `TimeAlive`.
- `AvgCentroidDistance`: distancia media respecto de un centroide.
- `AvgRoundVelocity`: velocidad media durante la ronda.
- `AvgKillDistance`: distancia media de las eliminaciones.
- `AvgSiteDistance`: distancia media respecto del sitio.
- `AvgMatchKillDist`: distancia media de eliminación a nivel de partida.

 `TimeAlive`, `TravelledDistance` y `FirstKillTime` aparecen como variables numéricas (`float64`) y son directamente analizables.

In [ ]:
print("Dimensiones:", df_raw.shape)
print("\nColumnas:")
print(df_raw.columns.tolist())

print("\nInformación del dataset:")
df_raw.info()

In [ ]:
# Tipo, número de valores únicos y cantidad de nulos por columna
resumen_estructura = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_unique": df_raw.nunique(dropna=True),
    "missing": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

display(resumen_estructura)

### Interpretación inicial del esquema

A partir del dataset se puede establecer preliminarmente que:

- la unidad original corresponde a **un jugador dentro de una ronda**;
- `Map` y `Team` son variables categóricas nominales;
- `RoundWinner`, `MatchWinner`, `Survived` y `AbnormalMatch` aparecen como booleanas;
- `TimeAlive`, `TravelledDistance`, `FirstKillTime` y las medidas de distancia/velocidad aparecen como variables numéricas continuas;
- `MatchId`, `InternalTeamId` y `SteamId` son identificadores y no deben interpretarse como magnitudes;
- `RoundId` funciona como identificador/posición temporal de la ronda dentro de una partida;
- `Unnamed: 0` corresponde a un índice exportado y es candidato a eliminación.

La presencia de `SteamId` permite identificar jugadores de manera persistente, pero no se utilizará directamente como una variable numérica predictora porque su valor entero no representa una magnitud ni un orden.

## 2.2 Verificar la granularidad del dataset

Antes de calcular probabilidades o construir targets derivados debemos saber qué representa una fila.

In [ ]:
cols_granularidad = [
    c for c in ["MatchId", "RoundId", "InternalTeamId", "SteamId", "Team"]
    if c in df_raw.columns
]

display(df_raw[cols_granularidad].head(20))

if {"MatchId", "RoundId", "InternalTeamId"}.issubset(df_raw.columns):
    jugadores_por_equipo_ronda = (
        df_raw.groupby(["MatchId", "RoundId", "InternalTeamId"])
        .size()
        .describe()
    )
    print("Filas por equipo/ronda:")
    display(jugadores_por_equipo_ronda)

if {"MatchId", "RoundId", "InternalTeamId", "SteamId"}.issubset(df_raw.columns):
    repeticion_jugador = (
        df_raw.groupby(["MatchId", "RoundId", "InternalTeamId"])["SteamId"]
        .nunique()
        .describe()
    )
    print("Jugadores únicos por equipo/ronda:")
    display(repeticion_jugador)

### Interpretación de la granularidad

La combinación `MatchId + RoundId + InternalTeamId` representa un equipo dentro de una ronda. En condiciones normales se esperan cinco `SteamId` distintos por grupo.

Esto confirma que muchas variables de resultado de equipo (`RoundWinner`, `MatchWinner`, `TeamStartingEquipmentValue`) se encuentran repetidas en las filas de los jugadores. Por esta razón, las probabilidades de victoria y el análisis de las primeras rondas deberán calcularse después de deduplicar o agregar a nivel equipo-ronda/equipo-partida.

## 2.3 Duplicados

In [ ]:
print("Filas completamente duplicadas:", df_raw.duplicated().sum())

if {"MatchId", "RoundId", "InternalTeamId"}.issubset(df_raw.columns):
    print("\nEjemplos de grupos equipo/ronda:")
    display(
        df_raw.groupby(["MatchId", "RoundId", "InternalTeamId"])
        .size()
        .value_counts()
        .sort_index()
    )

## 2.4 Variables categóricas y binarias

In [ ]:
categoricas = df_raw.select_dtypes(include=["object", "category"]).columns.tolist()
booleanas = df_raw.select_dtypes(include=["bool"]).columns.tolist()

print("Categóricas:", categoricas)
print("Booleanas:", booleanas)

for col in categoricas:
    print(f"\n--- {col} ---")
    display(df_raw[col].value_counts(dropna=False).head(20))

for col in booleanas:
    print(f"\n--- {col} ---")
    display(df_raw[col].value_counts(dropna=False, normalize=False))
    display((df_raw[col].value_counts(dropna=False, normalize=True) * 100).round(2))

## 2.5 Validación de tipos, semántica y posible data leakage

`TimeAlive`, `TravelledDistance` y `FirstKillTime` son variables numéricas. Por ello, el principal control de calidad consiste en determinar **qué representa cada variable y en qué momento de la partida se conoce**.

Esta distinción es crítica para evitar utilizar información futura como predictor.

In [ ]:
# Resumen de tipos y cardinalidad
perfil_variables = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_unique": df_raw.nunique(dropna=True),
    "missing": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean() * 100
})

display(perfil_variables)

### 2.5.1 Roles preliminares de las variables

A partir del nombre y estructura de los campos, se propone la siguiente clasificación inicial:

**Identificadores / agrupación**
- `Unnamed: 0`: índice exportado.
- `MatchId`: identifica la partida.
- `InternalTeamId`: identifica al equipo dentro de la partida.
- `SteamId`: identifica al jugador.

**Contexto conocido antes o al inicio de la ronda**
- `Map`
- `Team`
- `RoundId`
- `RoundStartingEquipmentValue`
- `TeamStartingEquipmentValue`

**Targets**
- `MatchWinner`: clasificación.
- `TimeAlive`: regresión.

**Información generada durante o después de la ronda**
- `RoundWinner`
- `Survived`
- `ScaledTimeAlive`
- `TravelledDistance`
- `AvgRoundVelocity`
- `AvgKillDistance`
- `AvgCentroidDistance`
- `AvgSiteDistance`
- `RLethalGrenadesThrown`
- `RNonLethalGrenadesThrown`
- `FirstKillTime`
- `RoundKills`
- `RoundAssists`
- `RoundHeadshots`
- `RoundFlankKills`

**Información de partida completa o potencialmente posterior al instante de predicción**
- `MatchKills`
- `MatchFlankKills`
- `MatchAssists`
- `MatchHeadshots`
- `AvgMatchKillDist`

Las variables `PrimaryAssaultRifle`, `PrimarySniperRifle`, `PrimaryHeavy`, `PrimarySMG` y `PrimaryPistol` requieren validación de su definición exacta antes de decidir si representan información disponible al inicio de la ronda o estadísticas generadas durante ella.

### 2.5.2 Auditoría de `TimeAlive` y variables derivadas

`TimeAlive` es interpretable como variable continua. Sin embargo, el dataset muestra que `ScaledTimeAlive` parece ser una transformación directa de `TimeAlive`.

Si esta relación se confirma en análisis, `ScaledTimeAlive` debe excluirse de los predictores de la regresión porque entregaría al modelo información derivada directamente del target.

In [ ]:
cols_time = [
    c for c in [
        "TimeAlive", "ScaledTimeAlive", "TravelledDistance",
        "AvgRoundVelocity", "FirstKillTime"
    ] if c in df_raw.columns
]

display(df_raw[cols_time].head(20))

if {"TimeAlive", "ScaledTimeAlive"}.issubset(df_raw.columns):
    print("Correlación TimeAlive vs ScaledTimeAlive:")
    display(df_raw[["TimeAlive", "ScaledTimeAlive"]].corr())

    # Hipótesis: ScaledTimeAlive = TimeAlive / máximo TimeAlive de la ronda.
    if {"MatchId", "RoundId"}.issubset(df_raw.columns):
        max_time_round = (
            df_raw.groupby(["MatchId", "RoundId"])["TimeAlive"]
            .transform("max")
        )
        scaled_calculado = df_raw["TimeAlive"] / max_time_round

        validos = (
            df_raw["ScaledTimeAlive"].notna()
            & scaled_calculado.notna()
        )

        coincidencia = np.isclose(
            df_raw.loc[validos, "ScaledTimeAlive"],
            scaled_calculado.loc[validos],
            rtol=1e-5,
            atol=1e-8
        )

        print(
            "Porcentaje compatible con TimeAlive / max(TimeAlive de la ronda):",
            round(coincidencia.mean() * 100, 4) if len(coincidencia) else np.nan,
            "%"
        )

In [ ]:
import numpy as np
import pandas as pd

# Máximo TimeAlive dentro de cada ronda de cada partida
max_timealive_ronda = (
    df_raw
    .groupby(["MatchId", "RoundId"])["TimeAlive"]
    .transform("max")
)

# Valor escalado que esperaríamos si:
# ScaledTimeAlive = TimeAlive / max(TimeAlive de esa ronda)
scaled_calculado = df_raw["TimeAlive"] / max_timealive_ronda

# Comparación con tolerancia para evitar problemas de precisión float
coincide = np.isclose(
    df_raw["ScaledTimeAlive"],
    scaled_calculado,
    rtol=1e-5,
    atol=1e-8,
    equal_nan=False
)

print(
    "Porcentaje de coincidencia:",
    round(coincide.mean() * 100, 4),
    "%"
)

print(
    "Coinciden:",
    coincide.sum(),
    "de",
    len(coincide),
    "registros"
)

# Mostrar algunos ejemplos
comparacion = pd.DataFrame({
    "MatchId": df_raw["MatchId"],
    "RoundId": df_raw["RoundId"],
    "TimeAlive": df_raw["TimeAlive"],
    "MaxTimeAliveRonda": max_timealive_ronda,
    "ScaledTimeAlive_real": df_raw["ScaledTimeAlive"],
    "ScaledTimeAlive_calculado": scaled_calculado,
    "Coincide": coincide
})

display(comparacion.head(20))

# Ver ejemplos donde NO coincide
display(
    comparacion[~comparacion["Coincide"]].head(20)
)

La correlación de Pearson entre `TimeAlive` y `ScaledTimeAlive` es de aproximadamente 0,35, por lo que al considerar todas las observaciones en conjunto no se aprecia una relación lineal fuerte entre ambas variables.

Sin embargo, al comprobar directamente la hipótesis de que `ScaledTimeAlive` corresponde a `TimeAlive` dividido por el máximo `TimeAlive` de cada ronda, se obtuvo una coincidencia de 99,43 %, equivalente a 78.707 de 79.157 registros.

Este resultado indica que `ScaledTimeAlive` es, en la gran mayoría de los casos, una transformación directa de `TimeAlive`. La correlación global relativamente baja se explica porque el valor utilizado como denominador cambia entre rondas, modificando la escala de la relación entre ambas variables.

Por lo tanto, dado que `TimeAlive` será utilizado como target del problema de regresión, `ScaledTimeAlive` se excluirá de las variables predictoras. Incorporarla entregaría al modelo información derivada directamente de la variable que se intenta predecir, produciendo data leakage y una evaluación artificialmente optimista del modelo.


### 2.5.3 Auditoría de identificadores y variables constantes

Las variables identificadoras son necesarias para agrupar correctamente los datos, pero no deben entrar al modelo como si fueran magnitudes continuas.

In [ ]:
perfil_cardinalidad = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_unique": df_raw.nunique(dropna=True),
    "unique_pct": df_raw.nunique(dropna=True) / len(df_raw) * 100,
    "missing": df_raw.isna().sum()
}).sort_values(["n_unique", "unique_pct"])

display(perfil_cardinalidad)

constantes = perfil_cardinalidad[perfil_cardinalidad["n_unique"] <= 1]
print("\nVariables constantes o sin variación:")
display(constantes)

ids_esperados = [
    c for c in ["Unnamed: 0", "MatchId", "InternalTeamId", "SteamId"]
    if c in df_raw.columns
]
print("\nIdentificadores esperados:")
display(perfil_cardinalidad.loc[ids_esperados])

### 2.5.4 Consistencia de variables repetidas por grupo

Algunas variables deberían ser constantes dentro de un grupo lógico. Esta comprobación permite detectar errores sin depender de una inspección manual.

In [ ]:
chequeos_consistencia = {}

if {"MatchId", "Map"}.issubset(df_raw.columns):
    chequeos_consistencia["Map por MatchId"] = (
        df_raw.groupby("MatchId")["Map"].nunique(dropna=True)
    )

if {"MatchId", "RoundId", "InternalTeamId", "Team"}.issubset(df_raw.columns):
    chequeos_consistencia["Team por equipo-ronda"] = (
        df_raw.groupby(
            ["MatchId", "RoundId", "InternalTeamId"]
        )["Team"].nunique(dropna=True)
    )

if {"MatchId", "RoundId", "InternalTeamId", "RoundWinner"}.issubset(df_raw.columns):
    chequeos_consistencia["RoundWinner por equipo-ronda"] = (
        df_raw.groupby(
            ["MatchId", "RoundId", "InternalTeamId"]
        )["RoundWinner"].nunique(dropna=True)
    )

if {"MatchId", "InternalTeamId", "MatchWinner"}.issubset(df_raw.columns):
    chequeos_consistencia["MatchWinner por equipo-partida"] = (
        df_raw.groupby(
            ["MatchId", "InternalTeamId"]
        )["MatchWinner"].nunique(dropna=True)
    )

for nombre, serie in chequeos_consistencia.items():
    inconsistentes = serie[serie > 1]
    print(f"\n{nombre}")
    print("Grupos inconsistentes:", len(inconsistentes))
    if len(inconsistentes):
        display(inconsistentes.head(20))

In [ ]:
# Identificar los grupos donde un mismo equipo-ronda
# contiene más de un valor de Team

team_por_grupo = (
    df_raw
    .groupby(["MatchId", "RoundId", "InternalTeamId"])["Team"]
    .nunique(dropna=True)
)

grupos_team_inconsistente = (
    team_por_grupo[team_por_grupo > 1]
    .reset_index()
)

print(
    "Cantidad de grupos inconsistentes:",
    len(grupos_team_inconsistente)
)

# Recuperar todas las filas pertenecientes a esos grupos
detalle_team_inconsistente = df_raw.merge(
    grupos_team_inconsistente[
        ["MatchId", "RoundId", "InternalTeamId"]
    ],
    on=["MatchId", "RoundId", "InternalTeamId"],
    how="inner"
)

columnas_revision = [
    c for c in [
        "MatchId",
        "RoundId",
        "InternalTeamId",
        "SteamId",
        "Team",
        "RoundWinner",
        "MatchWinner",
        "RoundStartingEquipmentValue",
        "TeamStartingEquipmentValue"
    ]
    if c in detalle_team_inconsistente.columns
]

display(
    detalle_team_inconsistente[columnas_revision]
    .sort_values(
        ["MatchId", "RoundId", "InternalTeamId", "Team"]
    )
)

In [ ]:
conteo_team_inconsistente = (
    detalle_team_inconsistente
    .groupby(
        ["MatchId", "RoundId", "InternalTeamId", "Team"],
        dropna=False
    )
    .size()
    .reset_index(name="n_jugadores")
)

display(conteo_team_inconsistente)

In [ ]:
# Resumir la distribución de jugadores por Team
# dentro de cada grupo inconsistente

patrones_team = (
    conteo_team_inconsistente
    .pivot_table(
        index=["MatchId", "RoundId", "InternalTeamId"],
        columns="Team",
        values="n_jugadores",
        fill_value=0
    )
    .reset_index()
)

display(patrones_team)

# Frecuencia de cada patrón
columnas_team = [
    c for c in ["Terrorist", "CounterTerrorist"]
    if c in patrones_team.columns
]

print("\nFrecuencia de patrones:")
display(
    patrones_team[columnas_team]
    .value_counts()
    .reset_index(name="n_grupos")
)

El análisis de consistencia de la variable `Team` identificó 18 combinaciones de `MatchId`, `RoundId` e `InternalTeamId` en las que aparecen simultáneamente las categorías `Terrorist` y `CounterTerrorist`.

Al revisar la distribución interna de estos grupos, se observó que todos contienen los cinco jugadores esperados y presentan una mayoría clara: 14 grupos tienen una distribución de 4 jugadores `Terrorist` y 1 `CounterTerrorist`, mientras que 4 grupos presentan el patrón inverso de 1 `Terrorist` y 4 `CounterTerrorist`.

Debido a que un mismo equipo no puede jugar simultáneamente en ambos lados dentro de una misma ronda, este patrón es consistente con errores aislados de etiquetado en la variable `Team`. La ausencia de casos ambiguos, como distribuciones 3–2, permite establecer una regla de corrección basada en la mayoría del grupo.

Por lo tanto, durante la fase de preparación de los datos se corregirá `Team` asignando a cada registro el valor mayoritario dentro de su combinación `MatchId`, `RoundId` e `InternalTeamId`. Esta transformación se considera una corrección de integridad basada en la estructura del dominio, y no una imputación estadística.


### 2.5.5 Missing values en su contexto

El tratamiento se decidirá utilizando el dataset completo. Antes de imputar, se cuantifican los valores faltantes y se determina si pueden recuperarse mediante información repetida dentro de los grupos lógicos.

In [ ]:
missing_25 = pd.DataFrame({
    "cantidad": df_raw.isna().sum(),
    "porcentaje": df_raw.isna().mean() * 100
})
missing_25 = missing_25[missing_25["cantidad"] > 0].sort_values(
    "porcentaje", ascending=False
)
display(missing_25)

# Mostrar contexto de variables clave sólo si presentan missing.
for col in ["Team", "RoundWinner", "MatchWinner"]:
    if col in df_raw.columns and df_raw[col].isna().any():
        print(f"\n### Missing en {col}")
        filas = df_raw[df_raw[col].isna()]
        display(filas.head(20))

### Conclusiones que ya pueden sostenerse

1. `TimeAlive`, `TravelledDistance` y `FirstKillTime` son numéricas y pueden incluirse en el análisis descriptivo.
2. `SteamId`, `MatchId` e `InternalTeamId` cumplen funciones de identificación/agrupación y no deben interpretarse como medidas numéricas.
3. `ScaledTimeAlive` es una variable sospechosa de derivarse directamente de `TimeAlive`; fue auditada y se excluirá de la regresión.
4. Las variables de movimiento y distancia permiten estudiar distribuciones, dispersión, outliers y correlaciones con mayor profundidad.
5. Para predecir `MatchWinner` en etapas tempranas, las estadísticas finales de ronda/partida no pueden utilizarse como predictores.
6. Para predecir `TimeAlive`, las variables generadas durante la ronda tampoco deben utilizarse si no se conocen al instante de realizar la predicción.

## 2.6 Estadística descriptiva de variables numéricas

In [ ]:
numericas = df_raw.select_dtypes(include=np.number).columns.tolist()

id_cols = [
    c for c in ["Unnamed: 0", "MatchId", "InternalTeamId", "SteamId", "RoundId"]
    if c in numericas
]

numericas_analisis = [
    c for c in numericas
    if c not in id_cols
]

print("Variables numéricas detectadas:", numericas)
print("Identificadores excluidos del resumen analítico:", id_cols)
print("Variables para análisis descriptivo:", numericas_analisis)

display(df_raw[numericas_analisis].describe().T)

### Clasificación semántica de variables numéricas

El tipo `int64` o `float64` de Python no define por sí solo el significado estadístico. Se separarán identificadores, variables binarias, conteos y medidas continuas.

Esta distinción será importante al elegir gráficos, métodos de outliers y escalamiento.

In [ ]:
roles_numericos = {
    "identificadores": [
        c for c in ["Unnamed: 0", "MatchId", "InternalTeamId", "SteamId", "RoundId"]
        if c in df_raw.columns
    ],
    "ordinal_temporal": [
        c for c in ["RoundId"]
        if c in df_raw.columns
    ],
    "conteos": [
        c for c in [
            "RLethalGrenadesThrown", "RNonLethalGrenadesThrown",
            "PrimaryPistol",
            "RoundKills", "RoundAssists", "RoundHeadshots",
            "RoundFlankKills", "MatchKills", "MatchFlankKills",
            "MatchAssists", "MatchHeadshots"
        ] if c in df_raw.columns
    ],
    "continuas": [
        c for c in [
            "TimeAlive", "ScaledTimeAlive", "AvgCentroidDistance",
            "TravelledDistance", "AvgRoundVelocity", "AvgKillDistance",
            "AvgSiteDistance", "FirstKillTime",
            "RoundStartingEquipmentValue", "TeamStartingEquipmentValue",
            "AvgMatchKillDist"
        ] if c in df_raw.columns
    ],
    "armas_requieren_validacion": [
        c for c in [
            "PrimaryAssaultRifle", "PrimarySniperRifle",
            "PrimaryHeavy", "PrimarySMG"
        ] if c in df_raw.columns
    ]
}

for rol, columnas in roles_numericos.items():
    print(f"{rol}: {columnas}")

In [ ]:
def resumen_estadistico(df, columnas):
    filas = []
    for col in columnas:
        x = df[col].dropna()
        if len(x) == 0:
            continue

        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)

        filas.append({
            "variable": col,
            "n": x.size,
            "media": x.mean(),
            "media_recortada_10%": trim_mean(x, 0.10) if x.size >= 10 else np.nan,
            "mediana": x.median(),
            "moda": x.mode().iloc[0] if not x.mode().empty else np.nan,
            "varianza": x.var(),
            "desv_std": x.std(),
            "min": x.min(),
            "q1": q1,
            "q3": q3,
            "IQR": q3 - q1,
            "max": x.max(),
            "rango": x.max() - x.min(),
            "skewness": x.skew()
        })

    return pd.DataFrame(filas).set_index("variable")

resumen_num = resumen_estadistico(df_raw, numericas_analisis)
display(resumen_num)


### Interpretación del análisis estadístico descriptivo

El resumen estadístico muestra que las 26 variables numéricas analizadas cuentan con 79.157 observaciones, por lo que no se observan valores faltantes en estas columnas. Sin embargo, existen diferencias importantes en sus distribuciones, escalas y naturaleza estadística que deberán considerarse durante la preparación de los datos.

#### `TimeAlive`

`TimeAlive`, definido como target del problema de regresión, presenta una media de 72,80 y una mediana de 69,24. La media recortada al 10 % es 69,95, valor considerablemente más cercano a la mediana que a la media original. Esto indica que los valores extremos superiores ejercen cierta influencia sobre el promedio.

El 50 % central de las observaciones se encuentra aproximadamente entre 47,08 y 92,65, con un IQR de 45,57. Sin embargo, el valor máximo alcanza 2.990,56 y la distribución presenta una asimetría positiva extremadamente elevada (`skewness = 26,97`).

Por lo tanto, aunque la mayor parte de los tiempos se concentra en un rango relativamente acotado, existen valores extremos muy alejados del comportamiento general. Estos registros deberán estudiarse antes de construir el modelo de regresión para determinar si corresponden a situaciones válidas del juego o a anomalías. Para describir la tendencia central de `TimeAlive`, la mediana y la media recortada resultan más robustas que la media simple.

#### Variables de movimiento y distancia

`TravelledDistance` presenta una media de aproximadamente 104.189 y una mediana de 86.016. Su media recortada baja hasta aproximadamente 89.897, lo que evidencia una influencia considerable de valores extremos. Esta observación se refuerza con una asimetría de 47,63 y un valor máximo de aproximadamente 39,6 millones.

Un fenómeno todavía más extremo aparece en `AvgRoundVelocity`, cuya mediana es 83,62 y media 84,35, pero cuyo máximo alcanza aproximadamente 24.112 y presenta una asimetría de 69,75. Esto indica que existe un número reducido de valores muy alejados del comportamiento general, que deberán ser inspeccionados individualmente antes de decidir cualquier eliminación o transformación.

`AvgCentroidDistance` también presenta asimetría positiva (`1,45`). Su media, aproximadamente 1,12 millones, es considerablemente superior a su mediana de aproximadamente 741 mil, mostrando nuevamente una cola hacia valores altos.

`AvgSiteDistance`, en comparación, presenta una distribución menos asimétrica (`skewness = 0,87`), aunque todavía existe una ligera cola hacia valores elevados.

Estos resultados muestran que varias variables continuas no presentan distribuciones aproximadamente normales y contienen valores extremos. Por ello, no sería adecuado utilizar automáticamente métodos basados en Z-score para todas ellas. El método IQR, acompañado de boxplots y análisis del dominio, será más apropiado como primera aproximación.

#### `AvgKillDistance` y los valores cero

`AvgKillDistance` tiene una característica especialmente relevante: su mediana y primer cuartil son iguales a cero, mientras que su media alcanza aproximadamente 288 mil y su tercer cuartil aproximadamente 360 mil. Además, presenta una asimetría positiva de 5,07.

Esto sugiere una importante concentración de observaciones en cero junto con una distribución positiva para el resto de los casos. Antes de interpretar estos ceros como valores bajos o faltantes será necesario confirmar su significado en el dataset. Una hipótesis plausible es que un valor cero corresponda a jugadores que no realizaron eliminaciones durante la ronda, en cuyo caso se trataría de un cero estructural y no de un missing value.

Un comportamiento similar deberá verificarse en `FirstKillTime`, cuya moda es cero, aunque su mediana es aproximadamente 54,77. En este caso también será necesario determinar qué representa exactamente el valor cero antes de realizar transformaciones.

#### Variables de conteo

Variables como `RoundKills`, `RoundAssists`, `RoundHeadshots` y `RoundFlankKills` presentan distribuciones fuertemente concentradas en valores pequeños y, en varios casos, una mediana y moda iguales a cero.

Por ejemplo, `RoundKills` presenta una media de 0,67 y mediana de 0, mientras que `RoundAssists`, `RoundHeadshots` y `RoundFlankKills` tienen primer y tercer cuartil iguales a cero o muy próximos a cero.

Estas variables corresponden a conteos discretos y naturalmente presentan distribuciones asimétricas. Por esta razón, un valor alto no debe interpretarse automáticamente como un error. En particular, cuando Q1 y Q3 son iguales a cero el IQR también es cero, por lo que aplicar mecánicamente la regla de 1,5 × IQR marcaría prácticamente cualquier valor positivo como outlier. Para estas variables será más apropiado utilizar límites definidos por la naturaleza del juego y analizar sus frecuencias.

#### Variables de armamento

`PrimaryAssaultRifle`, `PrimarySniperRifle`, `PrimaryHeavy`, `PrimarySMG` y `PrimaryPistol` sólo toman valores entre 0 y 1, por lo que deben interpretarse como variables binarias y no como variables continuas.

En este contexto, su media representa directamente la proporción de observaciones en las que la característica está presente. Aproximadamente un 67,54 % de los registros presenta `PrimaryAssaultRifle`, un 8,98 % `PrimarySniperRifle`, un 0,60 % `PrimaryHeavy`, un 0,70 % `PrimarySMG` y un 22,18 % `PrimaryPistol`.

Por tratarse de variables binarias, medidas como skewness, IQR o detección tradicional de outliers tienen poca utilidad para estas columnas. Su análisis debe realizarse principalmente mediante frecuencias y proporciones.

#### Economía inicial

Las variables económicas son especialmente relevantes para la pregunta de negocio del proyecto.

`RoundStartingEquipmentValue` presenta una media de aproximadamente 3.778, una mediana de 4.700 y un IQR comprendido entre 1.600 y 5.200. Su asimetría es negativa (`-0,54`).

Por su parte, `TeamStartingEquipmentValue` presenta una media aproximada de 18.890, una mediana de 22.550 y un rango intercuartílico entre 10.900 y 26.050, con una asimetría de `-0,74`.

En ambos casos la mediana es claramente superior a la media. Esto indica que las observaciones con valores económicos bajos generan una cola hacia la izquierda y reducen el promedio general. Este patrón es coherente con la existencia de distintos niveles de equipamiento entre rondas.

Estas variables resultan especialmente interesantes para el objetivo del proyecto, ya que permiten estudiar si los diferentes niveles económicos al comienzo de una ronda se relacionan con la capacidad de ganar las primeras rondas y, posteriormente, con la probabilidad de ganar la partida.

Será conveniente analizar sus histogramas y posiblemente construir categorías económicas posteriormente, pero los límites de dichas categorías deberán justificarse a partir de las distribuciones observadas o del conocimiento del dominio.

#### Estadísticas acumuladas de partida

`MatchKills`, `MatchAssists`, `MatchHeadshots` y `MatchFlankKills` presentan distribuciones con asimetría positiva moderada. Por ejemplo, `MatchKills` tiene una media de 8,51, mediana de 8 y máximo de 41, mientras que `MatchHeadshots` presenta una media de 3,86, mediana de 3 y máximo de 22.

Estos valores pueden resultar útiles para describir el comportamiento de los jugadores, pero no deben utilizarse como predictores de `MatchWinner` si el objetivo es realizar una predicción durante las primeras rondas, debido a que representan información acumulada durante o al finalizar la partida y producirían data leakage.

#### `ScaledTimeAlive`

`ScaledTimeAlive` presenta una mediana de 0,92, moda de 1 y tercer cuartil de 1. Su distribución tiene asimetría negativa (`-1,05`).

Sin embargo, el análisis anterior mostró que aproximadamente el 99,43 % de sus valores son compatibles con una transformación de `TimeAlive` basada en el máximo de la ronda. Por lo tanto, no debe interpretarse como una característica independiente y será excluida del futuro modelo de regresión debido al riesgo directo de data leakage.

### Conclusión general

El análisis descriptivo permite identificar tres grupos de variables con comportamientos estadísticos distintos. Las variables continuas de tiempo, distancia y velocidad presentan, en varios casos, distribuciones fuertemente asimétricas y valores extremos que deberán analizarse antes del modelado. Las variables de conteo tienen una alta concentración en cero y requieren un tratamiento diferente al de las variables continuas. Finalmente, las variables binarias deben analizarse principalmente mediante proporciones y no mediante técnicas tradicionales de detección de outliers.

Las variables económicas muestran una dispersión considerable y una distribución sesgada hacia valores económicos bajos, lo que las convierte en variables especialmente relevantes para estudiar la relación entre economía inicial, desempeño temprano y resultado de la partida.

Estos resultados justifican que el tratamiento de outliers, el escalamiento y las transformaciones posteriores se realicen de forma diferenciada según la naturaleza de cada variable, en lugar de aplicar una única técnica a todo el conjunto de datos.


## 2.7 Distribuciones

In [ ]:
variables_distribucion = [
    c for c in [
        "TimeAlive",
        "TravelledDistance",
        "AvgRoundVelocity",
        "AvgCentroidDistance",
        "AvgKillDistance",
        "AvgSiteDistance",
        "FirstKillTime",
        "RoundStartingEquipmentValue",
        "TeamStartingEquipmentValue",
        "RoundKills",
        "MatchKills",
        "AvgMatchKillDist"
    ]
    if c in df_raw.columns
]

for col in variables_distribucion:
    s = df_raw[col].dropna()

    if len(s) == 0:
        continue

    plt.figure(figsize=(8, 4))
    plt.hist(s, bins="auto")
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

In [ ]:
for col in variables_distribucion:
    plt.figure(figsize=(8, 2.5))
    sns.boxplot(x=df_raw[col])
    plt.title(f"Boxplot de {col}")
    plt.show()

### Interpretación
Para cada gráfico describir:

- forma de la distribución;
- asimetría;
- concentración de valores;
- posible presencia de outliers;
- si la variable parece adecuada para una transformación o escalamiento posterior.

## 2.8 Missing values

In [ ]:
missing = pd.DataFrame({
    "cantidad": df_raw.isna().sum(),
    "porcentaje": df_raw.isna().mean() * 100
})

missing = missing[missing["cantidad"] > 0].sort_values("porcentaje", ascending=False)
display(missing)

In [ ]:
# Revisar las filas con al menos un missing value.
filas_missing = df_raw[df_raw.isna().any(axis=1)]
print("Filas con al menos un missing:", len(filas_missing))
display(filas_missing.head(30))

### Interpretación

El dataset con contenía missing values de ninguna índole, es un dataset íntegro en ese sentido.

## 2.9 Detección de outliers — IQR

In [ ]:
def detectar_outliers_iqr(serie):
    s = pd.to_numeric(serie, errors="coerce").dropna()

    if len(s) == 0:
        return {
            "q1": np.nan, "q3": np.nan, "iqr": np.nan,
            "limite_inf": np.nan, "limite_sup": np.nan,
            "n_outliers": 0, "pct_outliers": 0
        }

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    mask = (serie < limite_inf) | (serie > limite_sup)

    return {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "limite_inf": limite_inf,
        "limite_sup": limite_sup,
        "n_outliers": int(mask.sum()),
        "pct_outliers": float(mask.mean() * 100)
    }

outlier_candidates = [
    c for c in [
        "TimeAlive", "TravelledDistance", "AvgRoundVelocity",
        "AvgCentroidDistance", "AvgKillDistance", "AvgSiteDistance",
        "FirstKillTime",
        "RoundStartingEquipmentValue", "TeamStartingEquipmentValue",
        "RLethalGrenadesThrown", "RNonLethalGrenadesThrown",
        "RoundKills", "RoundAssists", "RoundHeadshots",
        "RoundFlankKills", "MatchKills", "MatchFlankKills",
        "MatchAssists", "MatchHeadshots", "AvgMatchKillDist"
    ]
    if c in df_raw.columns
]

outliers_iqr = pd.DataFrame({
    col: detectar_outliers_iqr(df_raw[col])
    for col in outlier_candidates
}).T

display(outliers_iqr.sort_values("pct_outliers", ascending=False))

In [ ]:
# Mostrar los valores extremos de cada variable sin eliminarlos.
for col in outlier_candidates:
    info = outliers_iqr.loc[col]
    li = info["limite_inf"]
    ls = info["limite_sup"]

    extremos = df_raw.loc[
        (df_raw[col] < li) | (df_raw[col] > ls),
        col
    ]

    if len(extremos) > 0:
        print(f"\n{col}: {len(extremos)} valores marcados por IQR")
        display(extremos.value_counts().sort_index().tail(15))

### Distinción necesaria

Un *outlier estadístico* no equivale automáticamente a un error.

En variables discretas y acotadas, como `RoundKills`, obtener un valor poco frecuente de 4 o 5 puede ser perfectamente válido. Por ello, los límites IQR servirán para **detectar y revisar**, no para ejecutar una eliminación automática.

La decisión final deberá combinar:
- distribución observada;
- límites lógicos del juego;
- frecuencia;
- impacto sobre el análisis/modelo.

### Importante
Un valor marcado por IQR **no debe eliminarse automáticamente**.

Debemos distinguir:

1. **Error de datos:** valor imposible → corregir/eliminar.
2. **Valor extremo pero válido:** puede contener información importante → conservar o transformar.
3. **Valor extremo que afecta fuertemente un modelo:** evaluar cap/winsorización, transformación o modelo robusto.

En variables de conteo de videojuegos, por ejemplo, un número alto de kills puede ser raro pero totalmente real.

## 2.10 Z-score sólo en variables aproximadamente gaussianas

In [ ]:
# Primero seleccionar manualmente variables cuya distribución sea razonablemente simétrica.
variables_zscore = ["RoundStartingEquipmentValue", "TeamStartingEquipmentValue", "AvgSiteDistance"]

for col in variables_zscore:
    z = np.abs(stats.zscore(df_raw[col].dropna()))
    print(col, "outliers |z| > 3:", (z > 3).sum())
    
    datos = df_raw[col].dropna()

    print(f"\n### {col}")
    print(f"Media: {datos.mean():.4f}")
    print(f"Mediana: {datos.median():.4f}")
    print(f"Skewness: {datos.skew():.4f}")
    print(f"Kurtosis: {datos.kurtosis():.4f}")

    # Histograma
    plt.figure(figsize=(8, 4))
    plt.hist(datos, bins=40)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

    # Q-Q Plot
    plt.figure(figsize=(6, 6))
    stats.probplot(datos, dist="norm", plot=plt)
    plt.title(f"Q-Q Plot de {col}")
    plt.show()

### Interpretación

Ninguna de las variables es aproximadamente gaussiana, se descarta el uso de Z-score como método principal para detectar outliers.

Para cada una de las variables seleccionadas:
- El histograma no tiene forma de campana simétrica.
- En el Q-Q plot los puntos se separan marcadamente de la recta teórica

Para `AvgSiteDistance`:
- Los 656 registros con |z| > 3 indican observaciones alejadas de la media en términos de desviaciones estándar, pero como la distribución no es normal, eso no implica automáticamente que esos 656 casos sean verdaderos outliers.

## 2.11 Correlación entre variables numéricas

In [ ]:
corr_cols = [
    c for c in numericas_analisis
    if df_raw[c].nunique(dropna=True) > 1
]

corr = df_raw[corr_cols].corr(method="pearson")

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matriz de correlación de Pearson")
plt.show()

if "TimeAlive" in corr.columns:
    print("\nCorrelaciones lineales con TimeAlive:")
    display(
        corr["TimeAlive"]
        .drop("TimeAlive")
        .sort_values(key=lambda s: s.abs(), ascending=False)
        .to_frame("corr_TimeAlive")
    )

### Interpretación

El análisis de correlación de Pearson muestra que no existen variables con una relación lineal fuerte con TimeAlive. El coeficiente más elevado corresponde a ScaledTimeAlive (r = 0,3503), pero esta variable fue identificada previamente como una transformación directa de TimeAlive en aproximadamente el 99,43 % de los registros. Por esta razón será excluida del modelo de regresión debido a data leakage.

Entre las variables restantes, FirstKillTime presenta la mayor correlación positiva (r = 0,2568), seguida por RNonLethalGrenadesThrown (r = 0,2307). Sin embargo, ambas variables representan eventos que ocurren durante el desarrollo de la ronda. Si el objetivo es estimar TimeAlive utilizando únicamente información disponible al inicio de la ronda, tampoco pueden utilizarse como predictores.

Otras variables generadas durante la ronda, como TravelledDistance (r = 0,1537), AvgCentroidDistance (r = 0,1517), RLethalGrenadesThrown (r = 0,1491) y RoundKills (r = 0,1277), muestran asociaciones lineales débiles con el tiempo de supervivencia, pero deberán excluirse del modelo predictivo por la misma razón temporal.

Entre las variables potencialmente disponibles al comienzo de la ronda, se observan correlaciones débiles con TimeAlive. PrimaryPistol presenta una relación negativa de r = -0,1395, mientras que PrimaryAssaultRifle presenta una relación positiva de r = 0,1134. Asimismo, TeamStartingEquipmentValue (r = 0,1077) y RoundStartingEquipmentValue (r = 0,1043) muestran asociaciones positivas débiles. Esto sugiere que una mayor disponibilidad de equipamiento podría estar relacionada con una supervivencia ligeramente mayor, aunque la relación lineal individual es reducida.

La ausencia de correlaciones elevadas no implica que estas variables carezcan de capacidad predictiva. El tiempo de supervivencia puede depender de la interacción entre múltiples características, además de relaciones no lineales que el coeficiente de Pearson no captura. Por ello, en etapas posteriores será necesario evaluar modelos multivariables y comparar su desempeño mediante métricas de regresión.

Además, dado que TimeAlive presenta una distribución fuertemente asimétrica y valores extremos, los coeficientes de Pearson deben interpretarse con cautela, ya que esta métrica puede verse afectada por observaciones extremas.

## 2.12 Distribución de targets preliminares

In [ ]:
for target in [c for c in ["MatchWinner", "RoundWinner", "Survived"] if c in df_raw.columns]:
    print(f"\n### {target}")
    display(df_raw[target].value_counts(dropna=False))
    display((df_raw[target].value_counts(normalize=True, dropna=False) * 100).round(2))

if "TimeAlive" in df_raw.columns:
    print("\n### TimeAlive — target de regresión")
    display(df_raw["TimeAlive"].describe())

    s = df_raw["TimeAlive"].dropna()

    if len(s) > 0:
        plt.figure(figsize=(8, 4))
        plt.hist(s, bins="auto")
        plt.title("Distribución del target TimeAlive")
        plt.xlabel("TimeAlive")
        plt.ylabel("Frecuencia")
        plt.show()

        plt.figure(figsize=(8, 2.5))
        sns.boxplot(x=s)
        plt.title("Boxplot del target TimeAlive")
        plt.show()

## 2.13 Primer análisis de la hipótesis: rondas iniciales y victoria

Este análisis se realizará a nivel **equipo-ronda**, no a nivel jugador, para evitar contar varias veces el mismo resultado de ronda.

In [ ]:
required = {
    "MatchId", "RoundId", "InternalTeamId",
    "Team", "RoundWinner", "MatchWinner"
}

if required.issubset(df_raw.columns):
    team_round = (
        df_raw[
            [
                "MatchId", "RoundId", "InternalTeamId",
                "Team", "RoundWinner", "MatchWinner"
            ]
        ]
        .drop_duplicates()
        .sort_values(["MatchId", "InternalTeamId", "RoundId"])
        .reset_index(drop=True)
    )

    # Verificar que exista sólo una fila lógica por equipo-ronda.
    duplicados_logicos = (
        team_round.groupby(
            ["MatchId", "RoundId", "InternalTeamId"]
        ).size()
    )

    print(
        "Grupos equipo-ronda con más de una combinación lógica:",
        int((duplicados_logicos > 1).sum())
    )
    print("Filas equipo-ronda:", len(team_round))
    display(team_round.head(20))
else:
    print("Faltan columnas necesarias:", required - set(df_raw.columns))

### Nota metodológica
Para estudiar “las dos primeras rondas de cada lado” no debemos asumir que son simplemente RoundId 1-2 y 16-17.  
Primero debemos detectar, para cada `MatchId + InternalTeamId + Team`, cuáles son las primeras dos rondas efectivamente jugadas en ese lado.

In [ ]:
if "team_round" in globals():
    primeras_dos_por_lado = (
        team_round
        .sort_values(["MatchId", "InternalTeamId", "Team", "RoundId"])
        .groupby(["MatchId", "InternalTeamId", "Team"], group_keys=False)
        .head(2)
    )

    resumen_inicio = (
        primeras_dos_por_lado
        .groupby(["MatchId", "InternalTeamId", "Team"])
        .agg(
            rondas_observadas=("RoundId", "nunique"),
            primeras_dos_ganadas=("RoundWinner", "sum"),
            MatchWinner=("MatchWinner", "first")
        )
        .reset_index()
    )

    display(resumen_inicio.head(20))

In [ ]:
if "resumen_inicio" in globals():
    pivot_inicio = (
        resumen_inicio
        .pivot_table(
            index=["MatchId", "InternalTeamId"],
            columns="Team",
            values="primeras_dos_ganadas",
            aggfunc="first"
        )
        .reset_index()
    )

    ganador_match = (
        team_round
        .groupby(["MatchId", "InternalTeamId"], as_index=False)
        .agg(MatchWinner=("MatchWinner", "first"))
    )

    equipo_match = pivot_inicio.merge(
        ganador_match,
        on=["MatchId", "InternalTeamId"],
        how="left"
    )

    # Adaptable a los nombres reales de Team.
    if {"Terrorist", "CounterTerrorist"}.issubset(equipo_match.columns):
        equipo_match["OpeningRoundsWon"] = (
            equipo_match["Terrorist"].fillna(0)
            + equipo_match["CounterTerrorist"].fillna(0)
        )

        equipo_match["WonFirst2BothSides"] = (
            (equipo_match["Terrorist"] == 2)
            & (equipo_match["CounterTerrorist"] == 2)
        )

        display(equipo_match.head())
        print("\nProbabilidad de ganar según rondas iniciales ganadas:")
        display(
            equipo_match.groupby("OpeningRoundsWon")["MatchWinner"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "probabilidad_ganar"})
        )

        print("\nGanó las primeras 2 de ambos lados:")
        display(
            equipo_match.groupby("WonFirst2BothSides")["MatchWinner"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "probabilidad_ganar"})
        )

### Conclusión Fase 2 — estado actual

El análisis exploratorio permitió confirmar que el dataset tiene una granularidad principal jugador-ronda y contiene variables numéricas, categóricas, binarias, de conteo y continuas que requieren tratamientos diferenciados. Se detectaron además inconsistencias puntuales en Team, atribuibles a errores de etiquetado aislados, y se identificó ScaledTimeAlive como una variable derivada directamente de TimeAlive en el 99,43 % de los casos, por lo que deberá excluirse para evitar data leakage.

Desde el punto de vista estadístico, varias variables continuas presentan distribuciones asimétricas y valores extremos, por lo que no resulta adecuado asumir normalidad ni aplicar un único criterio de outliers a todo el conjunto. TimeAlive, definido como target de regresión, no presenta correlaciones lineales fuertes con variables disponibles al inicio de la ronda, lo que sugiere que su comportamiento probablemente depende de la combinación de múltiples factores.

En conjunto, los resultados obtenidos permiten avanzar hacia la preparación de los datos, priorizando la corrección de inconsistencias, el tratamiento diferenciado de outliers, el control de data leakage y la selección de variables según su significado y momento de disponibilidad.

# FASE 3 — PREPARACIÓN DE LOS DATOS

> Esta fase se completará **sólo después de interpretar la Fase 2**.  
> Las siguientes celdas son plantillas y deben activarse según lo que indiquen los resultados.

In [ ]:
df = df_raw.copy()

# 1) Índice exportado
cols_indice = [
    c for c in df.columns
    if c.lower().startswith("unnamed")
]

print("Columnas de índice detectadas:", cols_indice)

if cols_indice:
    df = df.drop(columns=cols_indice)

print("Dimensiones después de eliminar índice exportado:", df.shape)

## 3.1 Selección, limpieza e integridad

El nuevo dataset ya contiene correctamente tipadas las principales variables continuas. Por lo tanto, la preparación se enfocará en:

- eliminar artefactos de exportación;
- preservar identificadores sólo para agrupación y separación de datos;
- tratar missing values según su naturaleza;
- decidir el tratamiento de outliers;
- codificar variables categóricas;
- escalar variables cuando el algoritmo lo requiera;
- construir dos datasets analíticos separados para clasificación y regresión;
- excluir variables que generen *data leakage*.

## 3.1.1 Eliminación de artefactos e identificadores predictivos

`Unnamed: 0` es un índice exportado y se elimina. `MatchId`, `InternalTeamId` y `SteamId` se conservan para agrupar, validar y dividir los datos, pero no se utilizarán como magnitudes predictoras en los modelos base.

In [ ]:
df = df_raw.copy()

cols_indice = [
    c for c in df.columns
    if c.lower().startswith("unnamed")
]

if cols_indice:
    df = df.drop(columns=cols_indice)

print("Columnas de índice eliminadas:", cols_indice)
print("Dimensiones actuales:", df.shape)

## 3.1.2 Validación de tipos

En el nuevo sample:

- `TimeAlive`, `TravelledDistance`, `FirstKillTime` y las nuevas métricas continuas ya son `float64`.
- `RoundWinner`, `MatchWinner`, `Survived` y `AbnormalMatch` aparecen como booleanas.
- `Map` y `Team` son categóricas nominales.

El código siguiente vuelve a comprobar estos tipos sobre el dataset completo antes de continuar.

In [ ]:
columnas_clave = [
    c for c in [
        "Map", "Team", "RoundWinner", "MatchWinner",
        "TimeAlive", "TravelledDistance", "FirstKillTime",
        "ScaledTimeAlive", "AvgRoundVelocity"
    ]
    if c in df.columns
]

display(
    pd.DataFrame({
        "dtype": df[columnas_clave].dtypes.astype(str),
        "missing": df[columnas_clave].isna().sum(),
        "n_unique": df[columnas_clave].nunique(dropna=True)
    })
)

## 3.1.3 Exclusiones preliminares por data leakage

Se definen listas separadas para cada problema. Estas listas no eliminan todavía columnas del dataset original; documentan qué información es conceptualmente inválida para cada predicción.

In [ ]:
# Para predecir TimeAlive al inicio de una ronda:
leakage_regresion = [
    c for c in [
        "ScaledTimeAlive",
        "TravelledDistance",
        "AvgRoundVelocity",
        "AvgKillDistance",
        "AvgCentroidDistance",
        "AvgSiteDistance",
        "RLethalGrenadesThrown",
        "RNonLethalGrenadesThrown",
        "FirstKillTime",
        "RoundKills",
        "RoundAssists",
        "RoundHeadshots",
        "RoundFlankKills",
        "RoundWinner",
        "Survived",
        "MatchKills",
        "MatchFlankKills",
        "MatchAssists",
        "MatchHeadshots",
        "AvgMatchKillDist",
        "MatchWinner"
    ]
    if c in df.columns
]

# Para predecir MatchWinner usando información temprana:
leakage_clasificacion = [
    c for c in [
        "MatchKills",
        "MatchFlankKills",
        "MatchAssists",
        "MatchHeadshots",
        "AvgMatchKillDist"
    ]
    if c in df.columns
]

print("Variables excluidas preliminarmente de regresión por leakage:")
print(leakage_regresion)

print("\nVariables excluidas preliminarmente de clasificación por leakage:")
print(leakage_clasificacion)

## 3.1.4 Corregir Team usando el valor mayoritario dentro de cada equipo-ronda

In [ ]:
# Corregir Team usando el valor mayoritario dentro de cada equipo-ronda

def team_mayoritario(serie):
    moda = serie.mode()

    if len(moda) == 1:
        return pd.Series(
            [moda.iloc[0]] * len(serie),
            index=serie.index
        )

    # Si apareciera algún empate inesperado, mantener los valores originales
    return serie


df["Team_original"] = df["Team"]

df["Team"] = (
    df.groupby(
        ["MatchId", "RoundId", "InternalTeamId"],
        group_keys=False
    )["Team"]
    .apply(team_mayoritario)
)

# Comprobar cambios
cambios_team = df["Team_original"] != df["Team"]

print("Registros corregidos:", cambios_team.sum())

display(
    df.loc[
        cambios_team,
        [
            "MatchId",
            "RoundId",
            "InternalTeamId",
            "SteamId",
            "Team_original",
            "Team"
        ]
    ]
)

## 3.2 Estrategia de missing values

El tratamiento se escogerá según el tipo de variable, porcentaje de datos faltantes y mecanismo probable de pérdida.

Orden de decisión:

1. **Recuperación estructural**, si el valor puede obtenerse inequívocamente desde registros del mismo grupo.
2. **Eliminación de filas**, si el porcentaje es mínimo y no introduce un sesgo observable.
3. **Mediana**, para variables numéricas asimétricas o con outliers.
4. **Media**, sólo para variables aproximadamente simétricas y sin influencia fuerte de valores extremos.
5. **Moda/categoría explícita**, para variables categóricas cuando tenga sentido.
6. **KNN Imputer**, si existen missing numéricos suficientes, variables comparables y una noción de distancia útil.

La técnica no se seleccionará únicamente para demostrar su uso: debe estar justificada por los datos.

In [ ]:
# Comparación antes de decidir:
if len(missing) > 0:
    for col in missing.index:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"\n{col}")
            print("Media:", df[col].mean())
            print("Mediana:", df[col].median())
            print("N missing:", df[col].isna().sum())

## 3.3 KNN como estrategia de imputación — factibilidad

Con el nuevo dataset KNN puede ser una alternativa real si el dataset completo presenta missing values en variables continuas como distancias, velocidades o tiempos.

Sin embargo, KNN sólo se utilizará si:

- existe una cantidad de missing values que haga inconveniente eliminar filas;
- las columnas utilizadas para calcular vecinos están disponibles y son comparables;
- se eliminan identificadores;
- se codifican adecuadamente las variables categóricas utilizadas;
- las variables numéricas son escaladas, porque KNN se basa en distancias.

Si el dataset completo no presenta missing numéricos relevantes, KNN se documentará como alternativa evaluada pero no necesaria.

In [ ]:
# PLANTILLA: no ejecutar hasta seleccionar columnas adecuadas.
#
# from sklearn.impute import KNNImputer
# from sklearn.preprocessing import StandardScaler
#
# knn_cols = [...]
#
# X_knn = df[knn_cols].copy()
#
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_knn)
#
# imputer = KNNImputer(n_neighbors=5)
# X_imputed_scaled = imputer.fit_transform(X_scaled)
#
# X_imputed = scaler.inverse_transform(X_imputed_scaled)
# df_knn = df.copy()
# df_knn[knn_cols] = X_imputed

## 3.4 Tratamiento de outliers

No eliminar automáticamente todo lo detectado por IQR.

Para cada variable:

| Variable | Método detección | ¿Error o valor extremo válido? | Acción |
|---|---|---|---|
| ... | IQR / dominio | ... | conservar / corregir / eliminar / transformar |

Si una variable tiene valores extremos válidos, puede ser preferible conservarlos y evaluar transformaciones o modelos menos sensibles.

## 3.5 Encoding

`Map` y `Team` son variables categóricas nominales, por lo que no existe un orden natural entre sus categorías.

La técnica recomendada es **One-Hot Encoding**:

- `Map` → una columna binaria por mapa.
- `Team` → puede codificarse mediante One-Hot Encoding o como una variable binaria explícita.

Las variables booleanas (`RoundWinner`, `MatchWinner`, `Survived`, `AbnormalMatch`) pueden representarse como `0/1` cuando el algoritmo lo requiera.

No se utilizará Label Encoding para `Map`, ya que asignaría un orden artificial entre mapas.

`SteamId`, `MatchId` e `InternalTeamId` no requieren encoding para los modelos base porque serán utilizados como identificadores de agrupación, no como features.

In [ ]:
# Inspección previa a One-Hot Encoding
for col in [c for c in ["Map", "Team"] if c in df.columns]:
    print(col, "categorías:", df[col].nunique())
    display(df[col].value_counts())

In [ ]:
# Ejemplo pandas para análisis:
# df_encoded = pd.get_dummies(
#     df,
#     columns=["Map", "Team"],
#     drop_first=False,
#     dtype=int
# )
#
# display(df_encoded.head())

## 3.6 Escalamiento

El nuevo dataset contiene variables con escalas muy distintas:

- conteos pequeños como kills o granadas;
- `TimeAlive` en decenas de segundos;
- distancias con valores mucho mayores;
- valores de equipamiento en cientos o miles.

Por lo tanto, el escalamiento es **factible y relevante**, pero depende del algoritmo:

- **KNN / KNN Imputer:** necesario, porque utiliza distancia.
- **Ridge / Lasso:** recomendable, para que la penalización sea comparable entre variables.
- **Regresión lineal sin regularización:** no es obligatoria para obtener predicciones, aunque facilita comparar coeficientes.
- **Árboles de decisión / Random Forest:** normalmente no requieren escalamiento.

La selección entre `StandardScaler`, `MinMaxScaler` u otra alternativa se hará después de analizar distribuciones y outliers. `MinMaxScaler` es especialmente sensible a valores extremos.

In [ ]:
# Comparación de magnitudes antes de decidir escalamiento.
columnas_escalamiento = [
    c for c in [
        "RLethalGrenadesThrown", "RNonLethalGrenadesThrown",
        "PrimaryAssaultRifle", "PrimarySniperRifle",
        "PrimaryHeavy", "PrimarySMG",
        "RoundKills", "RoundAssists", "RoundHeadshots",
        "RoundFlankKills",
        "RoundStartingEquipmentValue",
        "TeamStartingEquipmentValue",
        "MatchKills", "MatchFlankKills",
        "MatchAssists", "MatchHeadshots"
    ]
    if c in df.columns
]

rango_escalas = pd.DataFrame({
    "min": df[columnas_escalamiento].min(),
    "max": df[columnas_escalamiento].max(),
    "media": df[columnas_escalamiento].mean(),
    "std": df[columnas_escalamiento].std(),
    "rango": (
        df[columnas_escalamiento].max()
        - df[columnas_escalamiento].min()
    )
}).sort_values("rango", ascending=False)

display(rango_escalas)

### Criterio de decisión

- Si se utiliza **KNN**, KNN Imputer, Ridge o Lasso, se recomienda escalar las variables predictoras numéricas.
- Si las distribuciones presentan outliers importantes, `MinMaxScaler` puede ser especialmente sensible a ellos.
- `StandardScaler` centra las variables y las expresa en unidades de desviación estándar, pero tampoco elimina el efecto de outliers extremos.
- Para modelos basados en árboles el escalamiento generalmente no es necesario.

La técnica concreta se escogerá después de observar distribuciones, asimetría y outliers.

In [ ]:
# Plantilla de comparación visual de escalamiento.
#
# from sklearn.preprocessing import MinMaxScaler, StandardScaler
#
# cols_scale = [...]
#
# minmax = MinMaxScaler()
# standard = StandardScaler()
#
# X_minmax = minmax.fit_transform(df[cols_scale])
# X_standard = standard.fit_transform(df[cols_scale])

## 3.7 Construcción de datasets analíticos

Debido a que clasificación y regresión utilizan diferentes unidades de observación, se construirán dos tablas separadas.

### A. Dataset de clasificación — equipo/partida

Target: `MatchWinner`.

Las features principales se derivarán de:

- mapa;
- primeras dos rondas jugadas como Terrorist;
- primeras dos rondas jugadas como Counter-Terrorist;
- cantidad de esas cuatro rondas ganadas;
- información económica disponible en esos momentos.

### B. Dataset de regresión — jugador/ronda

Target: `TimeAlive`.

Inicialmente sólo se utilizarán variables conocidas al inicio de la ronda, por ejemplo:

- `Map`;
- `Team`;
- `RoundId`;
- `RoundStartingEquipmentValue`;
- `TeamStartingEquipmentValue`.

Las variables de armas podrán agregarse únicamente después de confirmar que representan el equipamiento disponible al comienzo de la ronda y no información generada posteriormente.

In [ ]:
# Dataset base de regresión a nivel jugador-ronda.
regression_features_base = [
    c for c in [
        "Map", "Team", "RoundId",
        "RoundStartingEquipmentValue",
        "TeamStartingEquipmentValue"
    ]
    if c in df.columns
]

regression_cols = [
    c for c in [
        "MatchId", "SteamId",
        *regression_features_base,
        "TimeAlive"
    ]
    if c in df.columns
]

df_regression = df[regression_cols].copy()

print("Dataset base de regresión:")
print(df_regression.shape)
display(df_regression.head())

# El dataset de clasificación se construirá desde team_round,
# después de validar las primeras dos rondas de cada lado.
if "equipo_match" in globals():
    df_classification = equipo_match.copy()
    print("\nDataset preliminar de clasificación:")
    print(df_classification.shape)
    display(df_classification.head())

## 3.8 Prevención de data leakage y futura separación train/test

### Clasificación (`MatchWinner`)
No se utilizarán como predictores estadísticas calculadas al final de la partida ni resultados de rondas posteriores al momento en que se realiza la predicción.

### Regresión (`TimeAlive`)
No se utilizarán como predictores:

- `ScaledTimeAlive`, si se confirma que deriva directamente de `TimeAlive`;
- `Survived`;
- resultados y estadísticas generadas durante la ronda;
- variables que utilicen `TimeAlive` en su propia construcción;
- resultados finales de la partida.

### Separación de entrenamiento y prueba
En una fase posterior, el `train_test_split` no deberá realizarse aleatoriamente por fila. Las observaciones de una misma `MatchId` están relacionadas entre sí, por lo que una partida completa debe quedar en entrenamiento o en prueba, pero no en ambos conjuntos.

Para regresión también será necesario considerar que un mismo `SteamId` aparece en múltiples rondas, evitando que la evaluación se vuelva artificialmente optimista.

# 4. Decisiones finales de preparación — completar con el dataset completo

| Tema | Evidencia actual | Decisión preliminar |
|---|---|---|
| `TimeAlive` | Ahora es `float64` e interpretable | Mantener como target de regresión |
| `ScaledTimeAlive` | En el sample deriva directamente de `TimeAlive` | Auditar y excluir de regresión si se confirma |
| `Unnamed: 0` | Índice exportado | Eliminar |
| `SteamId` | Identificador de jugador | Usar para agrupación; no como magnitud |
| `MatchId` | Identificador de partida | Usar para agrupación y futura separación train/test |
| `InternalTeamId` | Identificador local de equipo | Usar para agrupación |
| `Map` | Categórica nominal | One-Hot Encoding |
| `Team` | Categórica nominal | One-Hot/binaria |
| Missing values | Pendiente dataset completo | Elegir estrategia según cantidad y mecanismo |
| Outliers | Pendiente EDA completo | Detectar con IQR/boxplots y validar con dominio |
| Escalamiento | Existen magnitudes muy diferentes | Aplicar si se usa KNN/Ridge/Lasso |
| Target clasificación | Binario | `MatchWinner` |
| Target regresión | Continuo | `TimeAlive` |
| Granularidad clasificación | Equipo-partida | Construir tabla agregada |
| Granularidad regresión | Jugador-ronda | Mantener tabla a nivel jugador |

# 5. Cierre de la Evaluación Parcial 1

Con el nuevo dataset ya se encuentra definida una base metodológica para las tres primeras fases de CRISP-DM.

La versión final de la entrega deberá concluir, utilizando los resultados del dataset completo:

1. cuáles son las distribuciones y estadísticos principales;
2. qué variables presentan missing values y cuál es su mecanismo probable;
3. qué valores son outliers estadísticos y cuáles corresponden a observaciones válidas;
4. qué variables necesitan encoding;
5. si el escalamiento se justifica para las técnicas futuras;
6. cuáles variables deben excluirse por data leakage;
7. cómo se construyen los datasets separados de clasificación y regresión.

La fase termina cuando ambos conjuntos de datos se encuentran comprendidos, depurados y listos para una posterior etapa de modelado.